# acai.ai_embedding — Test & Examples Notebook

Interactive test suite for the hexagonal embedding module.
Run each cell top-to-bottom. All adapter tests use mocked SDK clients — **no real API calls** unless credentials are present in `.env`.

Each adapter has a **standalone example** (`*_example.py`) that this notebook imports and runs, followed by mocked unit-style tests.

**Architecture under test:**

| Layer | Component | Description |
|-------|-----------|-------------|
| Port | `EmbedderPort` | Abstract `get_embedding` / `get_embeddings` interface |
| Domain | `EmbedderConfig` | Configuration value object with validation |
| Domain | Exceptions | `EmbeddingError` hierarchy (TextTooLong, ModelInvocation, Configuration) |
| Adapter | `OpenAILargeEmbedder` | OpenAI text-embedding-3-large |
| Adapter | `OpenAIAdaEmbedder` | OpenAI text-embedding-ada-002 |
| Adapter | `BedrockTitanEmbedder` | Amazon Bedrock Titan v1 |
| Adapter | `VoyageAIEmbedder` | Voyage AI embedding service |
| Factory | `create_embedder()` | Composition root |

**Per-adapter examples:**

| File | Adapter |
|------|---------|
| `openai_large_example.py` | OpenAI text-embedding-3-large |
| `openai_ada_example.py` | OpenAI text-embedding-ada-002 |
| `bedrock_titan_example.py` | Amazon Bedrock Titan |
| `voyageai_example.py` | Voyage AI |

## 0 — Setup & Imports

In [14]:
import sys, math
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import MagicMock, patch

# Ensure acai is importable
_project_root = Path.cwd()
while _project_root.name != "acai" and _project_root != _project_root.parent:
    _project_root = _project_root.parent
_shared_python = _project_root.parent  # …/shared/python
if str(_shared_python) not in sys.path:
    sys.path.insert(0, str(_shared_python))

from acai.logging import create_logger, LoggerConfig, LogLevel
from acai.ai_embedding import (
    create_embedder,
    EmbedderPort,
    EmbedderConfig,
    EmbeddingError,
    TextTooLongError,
    ModelInvocationError,
    ConfigurationError,
)
from acai.ai_embedding.adapters.outbound.voyageai_embedder import (
    VoyageAIEmbedder,
    VoyageAIConfig,
)
from acai.ai_embedding.adapters.outbound.openai_large_embedder import (
    OpenAILargeEmbedder,
    OpenAILargeConfig,
)
from acai.ai_embedding.adapters.outbound.openai_ada_embedder import (
    OpenAIAdaEmbedder,
    OpenAIAdaConfig,
)

# Per-adapter standalone examples
from acai.ai_embedding._example.openai_large_example import main as openai_large_demo
from acai.ai_embedding._example.openai_ada_example import main as openai_ada_demo
from acai.ai_embedding._example.bedrock_titan_example import main as bedrock_titan_demo
from acai.ai_embedding._example.voyageai_example import main as voyageai_demo

# Shared logger for all tests
_logger = create_logger(
    LoggerConfig(service_name="embedding-test", log_level=LogLevel.DEBUG)
)

# Test result tracker
_results: list[tuple[str, bool, str]] = []

def _record(name: str, passed: bool, detail: str = "") -> None:
    _results.append((name, passed, detail))
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}  {name}" + (f"  — {detail}" if detail else ""))

def _l2_norm(vec: list[float]) -> float:
    return math.sqrt(sum(x * x for x in vec))

def _make_voyage_embedder(**overrides) -> VoyageAIEmbedder:
    """Build a VoyageAIEmbedder with a mocked voyageai.Client."""
    defaults = {"api_key": "test-key", "model_name": "voyage-3-large"}
    defaults.update(overrides)
    cfg = VoyageAIConfig(**defaults)
    with patch.object(VoyageAIEmbedder, "_initialize_client"):
        embedder = VoyageAIEmbedder(logger=_logger, config=cfg)
    embedder.client = MagicMock()
    return embedder

print("Imports OK ✔")

Imports OK ✔


## 1 — EmbedderConfig Validation

The `EmbedderConfig` dataclass validates constraints in `__post_init__`. Invalid values raise `ConfigurationError`.

In [15]:
# Default values
cfg = EmbedderConfig()
_record("Config — default max_text_length", cfg.max_text_length == 8192)
_record("Config — default timeout_seconds", cfg.timeout_seconds == 30)
_record("Config — default retry_attempts", cfg.retry_attempts == 3)

# Custom values
cfg2 = EmbedderConfig(max_text_length=500, timeout_seconds=10, retry_attempts=5)
_record("Config — custom values accepted", cfg2.max_text_length == 500 and cfg2.timeout_seconds == 10)

# Negative max_text_length
caught = False
try:
    EmbedderConfig(max_text_length=-1)
except ConfigurationError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Config — negative max_text_length blocked", caught)

# Zero timeout
caught = False
try:
    EmbedderConfig(timeout_seconds=0)
except ConfigurationError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Config — zero timeout blocked", caught)

✅ PASS  Config — default max_text_length
✅ PASS  Config — default timeout_seconds
✅ PASS  Config — default retry_attempts
✅ PASS  Config — custom values accepted
  Caught: max_text_length must be positive
✅ PASS  Config — negative max_text_length blocked
  Caught: timeout_seconds must be positive
✅ PASS  Config — zero timeout blocked


## 2 — OpenAI Large Example (`openai_large_example.py`)

Runs the standalone adapter example — shows config, factory wiring, and live API call (if `OPENAI_API_KEY` is set).

In [16]:
openai_large_demo()

# Verify config
cfg = OpenAILargeConfig(openai_api_key="demo-key")
_record("OpenAILargeConfig — default model", cfg.model_name == "text-embedding-3-large")
_record("OpenAILargeConfig — max_batch_size", cfg.max_batch_size == 2048)
_record("OpenAILargeConfig — encoding_format", cfg.encoding_format == "float")

=== OpenAI Large — Config ===
  Model:           text-embedding-3-large
  Max text length: 8192
  Max batch size:  2048
  Encoding format: float

=== Factory ===
  create_embedder(logger, provider='openai_large', api_key='…')
  Returns: OpenAILargeEmbedder (text-embedding-3-large)

  [SKIP] OPENAI_API_KEY not set -- skipping live call
  Set it in .env or the environment to enable.

Done.
✅ PASS  OpenAILargeConfig — default model
✅ PASS  OpenAILargeConfig — max_batch_size
✅ PASS  OpenAILargeConfig — encoding_format


## 3 — OpenAI Ada Example (`openai_ada_example.py`)

Runs the standalone adapter example for the older ada-002 model.

In [17]:
openai_ada_demo()

# Verify config
cfg = OpenAIAdaConfig(openai_api_key="demo-key")
_record("OpenAIAdaConfig — default model", cfg.model_name == "text-embedding-ada-002")
_record("OpenAIAdaConfig — max_text_length", cfg.max_text_length == 8192)

=== OpenAI Ada — Config ===
  Model:           text-embedding-ada-002
  Max text length: 8192

=== Factory ===
  create_embedder(logger, provider='openai_ada', api_key='…')
  Returns: OpenAIAdaEmbedder (text-embedding-ada-002)

  [SKIP] OPENAI_API_KEY not set -- skipping live call
  Set it in .env or the environment to enable.

Done.
✅ PASS  OpenAIAdaConfig — default model
✅ PASS  OpenAIAdaConfig — max_text_length


## 4 — Bedrock Titan Example (`bedrock_titan_example.py`)

Runs the standalone adapter example for Amazon Bedrock Titan.

In [18]:
bedrock_titan_demo()

# Verify config (import may fail without boto3)
try:
    from acai.ai_embedding.adapters.outbound.bedrock_titan_embedder import (
        BedrockTitanEmbedder,
        BedrockTitanConfig,
    )
    cfg = BedrockTitanConfig()
    _record("BedrockTitanConfig — default region", cfg.bedrock_service_region == "eu-central-1")
    _record("BedrockTitanConfig — max_text_length", cfg.max_text_length == 8000)
    _record("BedrockTitanConfig — model_id", BedrockTitanEmbedder.MODEL_ID == "amazon.titan-embed-text-v1")
except ModuleNotFoundError:
    print("  ⏭ boto3 not installed — config tests skipped")
    _record("BedrockTitanConfig (skipped, no boto3)", True)

=== Bedrock Titan — Config ===
  Model ID:          amazon.titan-embed-text-v1
  Region:            eu-central-1
  Max text length:   8000
  Retry attempts:    3

=== Factory ===
  create_embedder(logger, provider='bedrock_titan')
  create_embedder(logger, provider='bedrock_titan', aws_profile='my-profile')
  Returns: BedrockTitanEmbedder (amazon.titan-embed-text-v1)

  [SKIP] AWS_PROFILE not set -- skipping live call
  Set it in .env or the environment to enable.

Done.
✅ PASS  BedrockTitanConfig — default region
✅ PASS  BedrockTitanConfig — max_text_length
✅ PASS  BedrockTitanConfig — model_id


## 5 — VoyageAI Example (`voyageai_example.py`)

Runs the standalone adapter example — shows config, normalization, and live API call (if `VOYAGEAI_API_KEY` is set).

In [19]:
voyageai_demo()

# Verify config
vcfg = VoyageAIConfig()
_record("VoyageAIConfig — default model", vcfg.model_name == "voyage-3-large")
_record("VoyageAIConfig — normalize enabled", vcfg.normalize is True)
_record("VoyageAIConfig — no input_type", vcfg.input_type is None)
_record("VoyageAIConfig — max_batch_size", vcfg.max_batch_size == 128)

# Custom model
vcfg2 = VoyageAIConfig(
    api_key="demo-key",
    model_name="voyage-3",
    normalize=True,
    input_type="document",
)
_record("VoyageAIConfig — custom model", vcfg2.model_name == "voyage-3")
_record("VoyageAIConfig — input_type set", vcfg2.input_type == "document")

# Inherits base validation
caught = False
try:
    VoyageAIConfig(max_text_length=-1)
except ConfigurationError:
    caught = True
_record("VoyageAIConfig — inherits base validation", caught)

2026-03-30 19:57:23,222 - INFO - Initialized VoyageAIEmbedder | model=voyage-3-large
2026-03-30 19:57:23,223 - DEBUG - Generating embeddings | count=1


=== VoyageAI — Config ===
  Model:      voyage-3
  Normalize:  True
  Input type: document
  Max batch:  128

=== Vector normalization ===
  Raw:        [3.0, 4.0]
  Normalized: [0.6, 0.8]
  L2 norm:    1.000000
  Zero vec:   [0.0, 0.0, 0.0]  (unchanged)

=== Factory ===
  create_embedder(logger, provider='voyageai', api_key='…')
  create_embedder(logger, provider='voyageai', api_key='…', model_name='voyage-3')
  Returns: VoyageAIEmbedder

=== Live API call ===


2026-03-30 19:57:23,453 - DEBUG - Successfully generated embeddings | count=1 dimension=1024
2026-03-30 19:57:23,453 - DEBUG - Generating embeddings | count=2


  Model:      voyage-3-large
  Dimension:  1024
  Normalized: True
  Input type: None
  Tokens:     7
  First 5:    [-0.058470904890515846, 0.06749418815212486, 0.03555175519873288, -0.054500657871221936, -0.01967076712155724]


2026-03-30 19:57:23,753 - DEBUG - Successfully generated embeddings | count=2 dimension=1024


  Batch:      2 results, dim=1024

Done.
✅ PASS  VoyageAIConfig — default model
✅ PASS  VoyageAIConfig — normalize enabled
✅ PASS  VoyageAIConfig — no input_type
✅ PASS  VoyageAIConfig — max_batch_size
✅ PASS  VoyageAIConfig — custom model
✅ PASS  VoyageAIConfig — input_type set
✅ PASS  VoyageAIConfig — inherits base validation


## 6 — Vector Normalization (pure math, no API)

The `VoyageAIEmbedder._normalize()` static method produces unit vectors. Zero vectors are returned unchanged.

In [20]:
# Classic 3-4-5 triangle
raw = [3.0, 4.0]
normalized = VoyageAIEmbedder._normalize(raw)
norm = _l2_norm(normalized)
_record("Normalize — unit vector", abs(norm - 1.0) < 1e-9, f"L2={norm:.9f}")
print(f"  Raw: {raw}  →  Normalized: {normalized}")

# Already normalized
already = [1.0, 0.0, 0.0]
result = VoyageAIEmbedder._normalize(already)
_record("Normalize — already unit", all(abs(a - b) < 1e-9 for a, b in zip(result, already)))

# Zero vector unchanged
zero = [0.0, 0.0, 0.0]
result = VoyageAIEmbedder._normalize(zero)
_record("Normalize — zero vector unchanged", result == [0.0, 0.0, 0.0])

# High-dimensional
hd = [float(i) for i in range(1, 129)]
hd_norm = VoyageAIEmbedder._normalize(hd)
_record("Normalize — 128-dim unit", abs(_l2_norm(hd_norm) - 1.0) < 1e-9)

# Negative values
neg = VoyageAIEmbedder._normalize([-3.0, 4.0])
_record("Normalize — negative values", abs(_l2_norm(neg) - 1.0) < 1e-9 and neg[0] < 0)

✅ PASS  Normalize — unit vector  — L2=1.000000000
  Raw: [3.0, 4.0]  →  Normalized: [0.6, 0.8]
✅ PASS  Normalize — already unit
✅ PASS  Normalize — zero vector unchanged
✅ PASS  Normalize — 128-dim unit
✅ PASS  Normalize — negative values


## 7 — Exception Hierarchy

All domain exceptions inherit from `EmbeddingError`, enabling coarse-grained and fine-grained error handling.

In [21]:
_record("Hierarchy — TextTooLongError ⊂ EmbeddingError", issubclass(TextTooLongError, EmbeddingError))
_record("Hierarchy — ModelInvocationError ⊂ EmbeddingError", issubclass(ModelInvocationError, EmbeddingError))
_record("Hierarchy — ConfigurationError ⊂ EmbeddingError", issubclass(ConfigurationError, EmbeddingError))

print(f"\nException tree:")
print(f"  EmbeddingError")
for cls in [TextTooLongError, ModelInvocationError, ConfigurationError]:
    print(f"    └─ {cls.__name__}")

✅ PASS  Hierarchy — TextTooLongError ⊂ EmbeddingError
✅ PASS  Hierarchy — ModelInvocationError ⊂ EmbeddingError
✅ PASS  Hierarchy — ConfigurationError ⊂ EmbeddingError

Exception tree:
  EmbeddingError
    └─ TextTooLongError
    └─ ModelInvocationError
    └─ ConfigurationError


## 8 — Input Validation

All adapters share the same validation logic: empty inputs, non-strings, text-too-long, and batch-too-large are rejected.

In [22]:
emb = _make_voyage_embedder(max_text_length=10, max_batch_size=2)

# Empty list
caught = False
try:
    emb.get_embeddings([])
except ValueError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Validation — empty list rejected", caught)

# Non-string input
caught = False
try:
    emb.get_embeddings([123])  # type: ignore
except ValueError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Validation — non-string rejected", caught)

# Empty string
caught = False
try:
    emb.get_embeddings([""])
except ValueError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Validation — empty string rejected", caught)

# Text too long
caught = False
try:
    emb.get_embeddings(["x" * 100])
except TextTooLongError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Validation — text too long rejected", caught)

# Batch too large
caught = False
try:
    emb.get_embeddings(["a", "b", "c"])
except ValueError as exc:
    caught = True
    print(f"  Caught: {exc}")
_record("Validation — oversized batch rejected", caught)

2026-03-30 19:57:23,772 - INFO - Initialized VoyageAIEmbedder | model=voyage-3-large


  Caught: Input must be a non-empty list of strings
✅ PASS  Validation — empty list rejected
  Caught: All inputs must be non-empty strings
✅ PASS  Validation — non-string rejected
  Caught: All inputs must be non-empty strings
✅ PASS  Validation — empty string rejected
  Caught: Input text length (100) exceeds maximum (10)
✅ PASS  Validation — text too long rejected
  Caught: Batch size (3) exceeds maximum (2)
✅ PASS  Validation — oversized batch rejected


## 9 — Port Contract

Every adapter returned by the factory must be an instance of `EmbedderPort`.

In [23]:
emb = _make_voyage_embedder()
_record("Port contract — VoyageAI is EmbedderPort", isinstance(emb, EmbedderPort))

with patch("acai.ai_embedding.adapters.outbound.openai_large_embedder.OpenAI"):
    emb_oai = create_embedder(_logger, provider="openai_large", api_key="k")
_record("Port contract — OpenAILarge is EmbedderPort", isinstance(emb_oai, EmbedderPort))

try:
    with patch("boto3.Session") as ms:
        ms.return_value = MagicMock()
        emb_bt = create_embedder(_logger, provider="bedrock_titan")
    _record("Port contract — BedrockTitan is EmbedderPort", isinstance(emb_bt, EmbedderPort))
except ModuleNotFoundError:
    print("  ⏭ boto3 not installed — skipping Bedrock Titan port check")
    _record("Port contract — BedrockTitan (skipped, no boto3)", True)

print(f"\nAll adapters implement EmbedderPort ✔")

2026-03-30 19:57:23,778 - INFO - Initialized VoyageAIEmbedder | model=voyage-3-large
2026-03-30 19:57:23,779 - INFO - Initialized OpenAILargeEmbedder | model=text-embedding-3-large
2026-03-30 19:57:23,780 - INFO - Initialized BedrockTitanEmbedder | region=eu-central-1


✅ PASS  Port contract — VoyageAI is EmbedderPort
✅ PASS  Port contract — OpenAILarge is EmbedderPort
✅ PASS  Port contract — BedrockTitan is EmbedderPort

All adapters implement EmbedderPort ✔


## Summary

In [24]:
passed = sum(1 for _, ok, _ in _results if ok)
failed = sum(1 for _, ok, _ in _results if not ok)
total  = len(_results)

print(f"\n{'='*60}")
print(f"  RESULTS: {passed}/{total} passed, {failed} failed")
print(f"{'='*60}\n")

for name, ok, detail in _results:
    status = "PASS" if ok else "FAIL"
    print(f"  [{status}] {name}" + (f"  — {detail}" if detail else ""))


  RESULTS: 37/37 passed, 0 failed

  [PASS] Config — default max_text_length
  [PASS] Config — default timeout_seconds
  [PASS] Config — default retry_attempts
  [PASS] Config — custom values accepted
  [PASS] Config — negative max_text_length blocked
  [PASS] Config — zero timeout blocked
  [PASS] OpenAILargeConfig — default model
  [PASS] OpenAILargeConfig — max_batch_size
  [PASS] OpenAILargeConfig — encoding_format
  [PASS] OpenAIAdaConfig — default model
  [PASS] OpenAIAdaConfig — max_text_length
  [PASS] BedrockTitanConfig — default region
  [PASS] BedrockTitanConfig — max_text_length
  [PASS] BedrockTitanConfig — model_id
  [PASS] VoyageAIConfig — default model
  [PASS] VoyageAIConfig — normalize enabled
  [PASS] VoyageAIConfig — no input_type
  [PASS] VoyageAIConfig — max_batch_size
  [PASS] VoyageAIConfig — custom model
  [PASS] VoyageAIConfig — input_type set
  [PASS] VoyageAIConfig — inherits base validation
  [PASS] Normalize — unit vector  — L2=1.000000000
  [PASS] Normal